# Overview

This notebook contains the data cleaning code for the **Net Overseas Migration** dataset.
Things to note:
1. This specific Excel file has multiple files within it (tabs), denoting the different years in the Reference period.
2. These "wafers" (as the ABS calls them) will need to be handled separately.
3. Luckily, each wafer follows an identical format so the same cleaning code can be used for each.
4. The plan will be to eventually combine the data from all wafers into a single DataFrame that contains all the information. This can then be saved to disk and used in another notebook for EDA and visualisation.

Since each wafer follows the same format, the cleaning steps will be the same. So we will figure out the cleaning steps for one of the wafers, and iteratively apply this to all wafers.

In [89]:
import pandas as pd
data = pd.read_excel('../data/net_overseas_migration_location_specific.xlsx', sheet_name=0)

data

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,Net Overseas Migration (1),NaN,NaN,NaN,NaN
1,Reference period by State of residence by Dire...,NaN,NaN,NaN,NaN
2,Counting: Persons,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN
4,Filters:,NaN,NaN,NaN,NaN
5,Default Summation,Persons ((x1)),NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN
7,2006,NaN,NaN,NaN,NaN
8,Direction of migration (2),NaN,Arrival,Departure,Total
9,NaN,State of residence,NaN,NaN,NaN


### Preprocessing steps to take
1. [x] Get rid of the header and footer rows (ABS metadata, not relevant to our analysis)
2. [x] Change the column and indices to what is correct (we need the columns to be the direction of migration and indices to be the states and territories)
3. [x] Get rid of any columns that are not needed. (metadata or Excel workbook formatting)

In [90]:
# we will define a cleaning function which takes in a raw dataframe and returns a cleaned version.
def clean_data(raw_data, print_output=False):
    raw_data_cleaned = raw_data.copy()
    data_year = raw_data_cleaned.iloc[7, 0]
    raw_data_cleaned.columns = raw_data_cleaned.loc[8]
    raw_data_cleaned = raw_data_cleaned.rename_axis('Direction of migration', axis='columns')
    raw_data_cleaned.index = raw_data_cleaned.iloc[:, 1]
    raw_data_cleaned = raw_data_cleaned.iloc[10:25, 2:]
    raw_data_cleaned['Arrival'] = raw_data_cleaned['Arrival'].astype(int)
    raw_data_cleaned['Departure'] = raw_data_cleaned['Departure'].astype(int)
    raw_data_cleaned['Total'] = raw_data_cleaned['Total'].astype(int)
    # this column will be important for when we merge DataFrames
    raw_data_cleaned['Year'] = data_year
    if print_output:
        print('=== DATASET INFORMATION ===')
        print('Dataset Type:', type(raw_data_cleaned))
        print('Data Shape:', raw_data_cleaned.shape)
        print()
        print('Data types present in dataset:')
        print(raw_data_cleaned.dtypes)
        print()
        print(raw_data_cleaned.describe())
    return raw_data_cleaned

## Phase 2: Combining all cleaned DataFrames into a single DataFrame that contains information across years.

In [91]:
# Sample output of the data from one of the sheets.
sample = clean_data(pd.read_excel('../data/net_overseas_migration_location_specific.xlsx', sheet_name=0))
sample

Direction of migration,Arrival,Departure,Total,Year
nan,,,,
New South Wales,75020,-39100,35920,2006
Victoria,51670,-23810,27860,2006
Queensland,41240,-21250,19980,2006
South Australia,11820,-4640,7180,2006
Western Australia,26430,-11520,14910,2006
Tasmania,1750,-950,800,2006
Northern Territory,2330,-2080,260,2006
Australian Capital Territory,3170,-2500,670,2006
Not Stated,0,0,0,2006


In [92]:
all_frames = []
for i in range(0, 20):
    frame_i = clean_data(pd.read_excel('../data/net_overseas_migration_location_specific.xlsx', sheet_name=i))
    all_frames.append(frame_i)

cleaned_data = pd.concat(all_frames)
cleaned_data['State-Territory'] = cleaned_data.index
cleaned_data.reset_index(drop=True, inplace=True)
cleaned_data

Direction of migration,Arrival,Departure,Total,Year,State-Territory
0,75020,-39100,35920,2006,New South Wales
1,51670,-23810,27860,2006,Victoria
2,41240,-21250,19980,2006,Queensland
3,11820,-4640,7180,2006,South Australia
4,26430,-11520,14910,2006,Western Australia
...,...,...,...,...,...
295,10,-10,10,2025,Christmas Island
296,0,0,0,2025,Jervis Bay
297,0,0,0,2025,Cocos (Keeling) Islands
298,0,0,0,2025,Other Territories


In [93]:
cleaned_data.groupby('Year')['Total'].sum()

Year
2006     215150
2007     488070
2008     631400
2009     493810
2010     344070
2011     412480
2012     480500
2013     416740
2014     364680
2015     373450
2016     487670
2017     483320
2018     504470
2019     495240
2020      -9930
2021      18610
2022     875730
2023    1061230
2024     659900
2025     320320
Name: Total, dtype: int64

In [94]:
cleaned_data.to_csv('../data/migration_data.csv')

# Summary
1. [x] Extracted each sheet from the Excel file.
2. [x] Got rid of all headers and footers, as well as unnecessary columns.
3. [x] Concatenated all the cleaned DataFrames together.
4. [x] Data is now ready to be analysed.